# CellMatch on NYU Cloud Bursting (HPC)

**Run all cells top to bottom** (`Run > Run All Cells`).

### Before you start
1. In [OOD Burst](https://ood.burst.hpc.nyu.edu): **Interactive Apps → JupyterLab**
2. Account: `cs_gy_6923-2026fa`
3. Partition with a GPU, e.g. `g2-standard-12` (L4) or `c12m85-a100-1` (A100)
4. Cores ≥ 4, Memory ≥ 24G, Time ≥ 8 hours
5. Open this notebook from `/scratch/ts5789/cellmatch/CellMatch_HPC.ipynb`

When Stage GPU + Stage CPU finish, `submission.csv` is in the project folder.


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil

WORK = Path("/scratch/ts5789/cellmatch")
assert WORK.exists(), f"Clone missing: {WORK}"
os.chdir(WORK)
print("WORK =", WORK.resolve())

# Make nested Slurm (if any) and relative paths consistent
os.environ["SLURM_SUBMIT_DIR"] = str(WORK)
os.environ["WORK_DIR"] = str(WORK)
os.environ["CELLPOSE_LOCAL_MODELS_PATH"] = str(WORK / "cellpose_weights")
os.environ["PIP_CACHE_DIR"] = str(WORK / ".pip_cache")
(WORK / "logs").mkdir(exist_ok=True)
(WORK / "cellpose_weights").mkdir(exist_ok=True)
(WORK / ".pip_cache").mkdir(exist_ok=True)
(WORK / "models").mkdir(exist_ok=True)


In [ ]:
# Bootstrap a project-local env (once). Uses scratch, not home (file-count limits).
import os, subprocess, sys
from pathlib import Path

WORK = Path("/scratch/ts5789/cellmatch").resolve()
CONDA = WORK / "miniforge"
ENV = WORK / "env"
PY = ENV / "bin" / "python"

def run(cmd, **kw):
    print("+", " ".join(map(str, cmd)), flush=True)
    subprocess.check_call(cmd, **kw)

if not PY.exists():
    if not (CONDA / "bin" / "conda").exists():
        sh = WORK / "miniforge.sh"
        run(["curl", "-fsSL", "-o", str(sh),
             "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh"])
        run(["bash", str(sh), "-b", "-p", str(CONDA)])
        sh.unlink(missing_ok=True)
    run([str(CONDA / "bin" / "conda"), "create", "-y", "-p", str(ENV), "python=3.12"])
    run([str(PY), "-m", "pip", "install", "--upgrade", "pip"])
    run([str(PY), "-m", "pip", "install",
         "torch", "torchvision", "tifffile", "cellpose>=4.2",
         "scikit-image", "scipy", "pandas", "opencv-python-headless", "ipykernel"])
    run([str(PY), "-m", "ipykernel", "install", "--user",
         "--name", "cellmatch", "--display-name", "Python (cellmatch)"])
    print("Kernel 'Python (cellmatch)' installed. If this notebook is not using it yet,")
    print("Kernel → Change Kernel → Python (cellmatch), then re-run from the top.")
else:
    print("env ready:", PY)

# Prefer project env for the rest of this notebook's shell magics
os.environ["PATH"] = f"{ENV/'bin'}:{os.environ.get('PATH','')}"
sys.path.insert(0, str(WORK))
print("python:", PY)


In [ ]:
import subprocess, sys
from pathlib import Path
PY = Path("/scratch/ts5789/cellmatch/env/bin/python")

out = subprocess.check_output(
    [str(PY), "-c",
     "import torch; print(torch.__version__); print('cuda', torch.cuda.is_available());"
     " print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO_GPU')"],
    text=True)
print(out)
assert "cuda True" in out, (
    "This Jupyter session has no GPU. Delete this session in OOD and start JupyterLab "
    "on partition g2-standard-12 or c12m85-a100-1, then reopen this notebook."
)
subprocess.check_call(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"])


In [ ]:
from pathlib import Path
DATA = Path("/scratch/ts5789/cellmatch/Project_2_Dataset")
assert (DATA / "training/train_ground_truth.csv").exists(), "Dataset missing under Project_2_Dataset"
print("training:", sorted(p.name for p in (DATA / "training").glob("subject_*")))
print("test regions:", len(list((DATA / "hidden_test").glob("*/*"))))


In [ ]:
# Full GPU stage: fold training, Cellpose fine-tunes, held-out + test prediction caches.
# Skips anything already under cache/ and models/.
from pathlib import Path
import subprocess
PY = Path("/scratch/ts5789/cellmatch/env/bin/python")
subprocess.check_call([str(PY), "run_hpc.py", "gpu"], cwd="/scratch/ts5789/cellmatch")


In [ ]:
# CPU stage on this same node: leak-free search + write submission.csv
from pathlib import Path
import subprocess
PY = Path("/scratch/ts5789/cellmatch/env/bin/python")
subprocess.check_call([str(PY), "run_hpc.py", "cpu"], cwd="/scratch/ts5789/cellmatch")


In [ ]:
from pathlib import Path
import subprocess
WORK = Path("/scratch/ts5789/cellmatch")
PY = WORK / "env" / "bin" / "python"
sub = WORK / "submission.csv"
assert sub.exists(), "submission.csv was not written"
subprocess.check_call([str(PY), "validate_submission.py", str(sub)], cwd=str(WORK))
print("DONE →", sub, f"({sub.stat().st_size:,} bytes)")
print(sub.read_text().splitlines()[0])
print("...", len(sub.read_text().splitlines()), "lines")
